# 02_16 Kaggle MedNeXt-S PHE-only baseline

Notebook nay de chay tren Kaggle, cung tinh than voi `02_12b` nhung khong dung nnU-Net trainer. Model duoc lay tu official repo `MIC-DKFZ/MedNeXt`.

Muc tieu:

```text
input  channel 0 = CT only
target label     = manual PHE mask
model            = official MIC-DKFZ MedNeXt-S
```

Notebook nay khong dung ICH teacher, khong dung ICH prior, va dung cung hardcoded split 99/10/11 de so sanh cong bang voi `02_12`, `02_12b`, `02_14`, `02_15`.


In [2]:
from pathlib import Path
import os
import sys
import re
import json
import math
import time
import random
import gc
import subprocess

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

# Optional manual overrides. Neu auto-detect sai, gan path nay tren Kaggle.
# Example:
# USER_PHE_ROOT = Path('/kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
# USER_MEDNEXT_REPO = Path('/kaggle/input/mednext/MedNeXt')
USER_PHE_ROOT = None
USER_MEDNEXT_REPO = None
MEDNEXT_REPO_URL = 'https://github.com/MIC-DKFZ/MedNeXt.git'

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_16_kaggle_mednext_s_phe_only_baseline'
PRED_DIR = OUTPUT_ROOT / 'predictions_mednext_s_best'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
for p in [OUTPUT_ROOT, PRED_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}

# Training defaults: gan voi 02_12b nhung co the giam neu Kaggle het time.
MAX_EPOCHS = 120
TRAIN_PATCHES_PER_EPOCH = 128
PATCH_SIZE = (32, 128, 128)  # z, y, x. De 128 cho an toan hon tren Kaggle T4.
BATCH_SIZE = 1
FG_PATCH_PROB = 0.70
VALIDATE_EVERY = 10
NUM_WORKERS = 0

# Official MedNeXt-S config. model_id S la Small theo MIC-DKFZ/MedNeXt.
MEDNEXT_MODEL_ID = 'S'
MEDNEXT_KERNEL_SIZE = 3
MEDNEXT_DEEP_SUPERVISION = False

LR = 3e-4
WEIGHT_DECAY = 1e-5
USE_AMP = True
GRAD_CLIP_NORM = 12.0

# CT-only baseline. Khong dung ICH. Window co the doi neu muon test multi-window sau nay.
CT_WINDOW = (-100.0, 200.0)
THRESHOLD = 0.5
POSTPROCESS_MIN_COMPONENT_SIZE = 0  # dat 50 neu muon thu paper-style postprocess
SLIDING_WINDOW_OVERLAP = 0.5

print('IS_KAGGLE   :', IS_KAGGLE)
print('WORK_ROOT   :', WORK_ROOT)
print('OUTPUT_ROOT :', OUTPUT_ROOT)
print('MAX_EPOCHS  :', MAX_EPOCHS)
print('PATCH_SIZE  :', PATCH_SIZE)


IS_KAGGLE   : True
WORK_ROOT   : /kaggle/working
OUTPUT_ROOT : /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline
MAX_EPOCHS  : 120
PATCH_SIZE  : (32, 128, 128)


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass

seed_everything(SEED)


def ensure_import(import_name, pip_name=None):
    import importlib
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

sitk = ensure_import('SimpleITK')
ensure_import('scipy')

import SimpleITK as sitk
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler



def has_mednext_repo(path_like):
    p = Path(path_like)
    return (p / 'nnunet_mednext').exists() and (p / 'setup.py').exists()


def find_mednext_repo():
    if USER_MEDNEXT_REPO is not None:
        p = Path(USER_MEDNEXT_REPO)
        if has_mednext_repo(p):
            return p
        raise FileNotFoundError(f'USER_MEDNEXT_REPO is not a MedNeXt repo: {p}')

    candidates = [LOCAL_ROOT / 'MedNeXt', LOCAL_ROOT / 'mednext']
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([base, base / 'MedNeXt', base / 'mednext'])
    for p in candidates:
        if has_mednext_repo(p):
            return p

    clone_dir = WORK_ROOT / 'MedNeXt'
    if not has_mednext_repo(clone_dir):
        print('Cloning official MedNeXt repo:', MEDNEXT_REPO_URL)
        subprocess.check_call(['git', 'clone', '--depth', '1', MEDNEXT_REPO_URL, str(clone_dir)])
    return clone_dir


MEDNEXT_REPO = find_mednext_repo()
if str(MEDNEXT_REPO) not in sys.path:
    sys.path.insert(0, str(MEDNEXT_REPO))

# Install editable without deps because we only use the architecture. This avoids pulling nnU-Net-v1 deps unnecessarily.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(MEDNEXT_REPO), '--no-deps'])

from nnunet_mednext import create_mednext_v1
print('Official MedNeXt repo:', MEDNEXT_REPO)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


Cloning official MedNeXt repo: https://github.com/MIC-DKFZ/MedNeXt.git


Cloning into '/kaggle/working/MedNeXt'...


Obtaining file:///kaggle/working/MedNeXt
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for mednextv1
Official MedNeXt repo: /kaggle/working/MedNeXt
torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [4]:
def strip_nii_suffix(path_like):
    name = Path(str(path_like)).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    if name.endswith('.nii'):
        return name[:-4]
    return Path(name).stem


def scan_id_from_name(path_like):
    stem = strip_nii_suffix(path_like)
    m = re.search(r'(\d{4})$', stem)
    if not m:
        m = re.search(r'(\d+)$', stem)
    if not m:
        raise ValueError(f'Cannot parse scan id from {path_like}')
    return m.group(1).zfill(4)


def has_nifti_pair_root(path_like):
    p = Path(path_like)
    if not (p / 'set').exists() or not (p / 'label').exists():
        return False
    image_count = len(list((p / 'set').glob('*.nii'))) + len(list((p / 'set').glob('*.nii.gz')))
    mask_count = len(list((p / 'label').glob('*.nii'))) + len(list((p / 'label').glob('*.nii.gz')))
    return image_count > 0 and mask_count > 0


def find_phe_root():
    if USER_PHE_ROOT is not None:
        p = Path(USER_PHE_ROOT)
        if has_nifti_pair_root(p):
            return p
        raise FileNotFoundError(f'USER_PHE_ROOT does not contain set/label NIfTI pairs: {p}')

    candidates = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        LOCAL_ROOT / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    if IS_KAGGLE:
        for base in KAGGLE_INPUT.iterdir():
            candidates.extend([
                base / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'SubdatasetA_NIFIT' / 'NIFIT',
                base / 'NIFIT',
                base,
            ])
    for p in candidates:
        if has_nifti_pair_root(p):
            return p

    search_roots = [KAGGLE_INPUT] if IS_KAGGLE else [LOCAL_ROOT]
    found = []
    for root in search_roots:
        for set_dir in root.rglob('set'):
            p = set_dir.parent
            if has_nifti_pair_root(p):
                found.append(p)
    if found:
        found = sorted(found, key=lambda p: (('SubdatasetA_NIFIT' not in str(p)), len(str(p))))
        return found[0]
    raise FileNotFoundError('Could not find PHE NIfTI root with set/label. Set USER_PHE_ROOT manually.')


PHE_ROOT = find_phe_root()
PHE_IMAGE_DIR = PHE_ROOT / 'set'
PHE_MASK_DIR = PHE_ROOT / 'label'

image_files = sorted(list(PHE_IMAGE_DIR.glob('*.nii')) + list(PHE_IMAGE_DIR.glob('*.nii.gz')))
mask_files = sorted(list(PHE_MASK_DIR.glob('*.nii')) + list(PHE_MASK_DIR.glob('*.nii.gz')))
mask_by_scan = {scan_id_from_name(p): p for p in mask_files}

rows = []
for image_path in image_files:
    scan_id = scan_id_from_name(image_path)
    mask_path = mask_by_scan.get(scan_id)
    if mask_path is None:
        continue
    if scan_id in TEST_SCAN_IDS:
        split = 'test'
    elif scan_id in VAL_SCAN_IDS:
        split = 'val'
    else:
        split = 'train'
    rows.append({
        'scan_id': scan_id,
        'case_id': f'phe_{scan_id}',
        'split': split,
        'image_path': str(image_path),
        'label_path': str(mask_path),
    })

manifest_df = pd.DataFrame(rows).sort_values(['split', 'scan_id']).reset_index(drop=True)
manifest_csv = OUTPUT_ROOT / '02_16_mednext_s_phe_only_manifest.csv'
manifest_df.to_csv(manifest_csv, index=False)

print('PHE root :', PHE_ROOT)
print('Images   :', len(image_files))
print('Masks    :', len(mask_files))
print('Matched  :', len(manifest_df))
print(manifest_df['split'].value_counts().to_dict())
print('Manifest :', manifest_csv)
display(manifest_df.groupby('split').head(3))


PHE root : /kaggle/input/datasets/dovune/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT
Images   : 120
Masks    : 120
Matched  : 120
{'train': 99, 'test': 11, 'val': 10}
Manifest : /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/02_16_mednext_s_phe_only_manifest.csv


,scan_id,case_id,split,image_path,label_path
0,0002,phe_0002,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
1,0029,phe_0029,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
2,0033,phe_0033,test,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
11,0001,phe_0001,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
12,0004,phe_0004,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
13,0005,phe_0005,train,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
110,0013,phe_0013,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
111,0014,phe_0014,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...
112,0060,phe_0060,val,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...,/kaggle/input/datasets/dovune/phe-sich-ct-ids/...


In [5]:
def read_nifti(path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return arr, img


def write_nifti_like(arr_zyx, reference_img, out_path):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_img = sitk.GetImageFromArray(arr_zyx.astype(np.uint8))
    out_img.CopyInformation(reference_img)
    sitk.WriteImage(out_img, str(out_path))


def normalize_ct(arr, window=CT_WINDOW):
    arr = arr.astype(np.float32)
    lo, hi = window
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / max(hi - lo, 1e-6)
    arr = arr * 2.0 - 1.0
    return arr.astype(np.float32)


def crop_with_pad(arr, center_zyx, patch_size, pad_value=0):
    patch_size = np.asarray(patch_size, dtype=np.int64)
    center = np.asarray(center_zyx, dtype=np.int64)
    start = center - patch_size // 2
    end = start + patch_size

    src_start = np.maximum(start, 0)
    src_end = np.minimum(end, np.asarray(arr.shape, dtype=np.int64))
    dst_start = src_start - start
    dst_end = dst_start + (src_end - src_start)

    out = np.full(tuple(patch_size), pad_value, dtype=arr.dtype)
    out[
        dst_start[0]:dst_end[0],
        dst_start[1]:dst_end[1],
        dst_start[2]:dst_end[2],
    ] = arr[
        src_start[0]:src_end[0],
        src_start[1]:src_end[1],
        src_start[2]:src_end[2],
    ]
    return out


def pad_to_at_least(arr, min_shape, pad_value=0):
    min_shape = np.asarray(min_shape, dtype=np.int64)
    shape = np.asarray(arr.shape, dtype=np.int64)
    pad_after = np.maximum(min_shape - shape, 0)
    pads = [(0, int(v)) for v in pad_after]
    if any(v > 0 for v in pad_after):
        arr = np.pad(arr, pads, mode='constant', constant_values=pad_value)
    return arr, tuple(int(v) for v in pad_after)


def dice_binary(pred, ref, eps=1e-8):
    pred = pred.astype(bool)
    ref = ref.astype(bool)
    inter = np.logical_and(pred, ref).sum(dtype=np.float64)
    denom = pred.sum(dtype=np.float64) + ref.sum(dtype=np.float64)
    if denom == 0:
        return 1.0
    return float((2.0 * inter + eps) / (denom + eps))


def metric_binary(pred, ref):
    pred = pred.astype(bool)
    ref = ref.astype(bool)
    tp = int(np.logical_and(pred, ref).sum())
    fp = int(np.logical_and(pred, ~ref).sum())
    fn = int(np.logical_and(~pred, ref).sum())
    tn = int(np.logical_and(~pred, ~ref).sum())
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 1.0
    return {
        'Dice': float(dice),
        'FN': fn,
        'FP': fp,
        'IoU': float(iou),
        'TN': tn,
        'TP': tp,
        'n_pred': int(pred.sum()),
        'n_ref': int(ref.sum()),
    }


def remove_small_components(mask, min_size=0):
    if min_size <= 0:
        return mask.astype(np.uint8)
    labeled, n = ndi.label(mask > 0)
    if n == 0:
        return mask.astype(np.uint8)
    counts = np.bincount(labeled.ravel())
    keep = np.zeros_like(counts, dtype=bool)
    keep[np.where(counts >= min_size)[0]] = True
    keep[0] = False
    return keep[labeled].astype(np.uint8)


In [6]:
class VolumeCache:
    def __init__(self, cache=True):
        self.cache = cache
        self._cache = {}

    def get(self, row):
        case_id = row['case_id']
        if self.cache and case_id in self._cache:
            return self._cache[case_id]

        image_arr, image_obj = read_nifti(row['image_path'])
        label_arr, _ = read_nifti(row['label_path'])
        image_arr = normalize_ct(image_arr)
        label_arr = (label_arr > 0).astype(np.uint8)
        fg_voxels = np.argwhere(label_arr > 0)
        item = {
            'case_id': case_id,
            'image': image_arr,
            'label': label_arr,
            'image_obj': image_obj,
            'fg_voxels': fg_voxels,
        }
        if self.cache:
            self._cache[case_id] = item
        return item


class RandomPatchDataset(Dataset):
    def __init__(self, rows_df, cache, patches_per_epoch, patch_size=PATCH_SIZE, fg_patch_prob=FG_PATCH_PROB):
        self.rows = rows_df.reset_index(drop=True).to_dict('records')
        self.cache = cache
        self.patches_per_epoch = int(patches_per_epoch)
        self.patch_size = tuple(int(x) for x in patch_size)
        self.fg_patch_prob = float(fg_patch_prob)

    def __len__(self):
        return self.patches_per_epoch

    def __getitem__(self, idx):
        row = random.choice(self.rows)
        item = self.cache.get(row)
        img = item['image']
        lab = item['label']
        fg = item['fg_voxels']

        if len(fg) > 0 and random.random() < self.fg_patch_prob:
            center = fg[random.randrange(len(fg))]
            jitter = np.array([random.randint(-8, 8), random.randint(-48, 48), random.randint(-48, 48)])
            center = center + jitter
        else:
            center = np.array([random.randrange(s) for s in img.shape])

        x = crop_with_pad(img, center, self.patch_size, pad_value=-1.0)
        y = crop_with_pad(lab, center, self.patch_size, pad_value=0)

        x = np.ascontiguousarray(x[None], dtype=np.float32)
        y = np.ascontiguousarray(y[None], dtype=np.float32)
        return torch.from_numpy(x), torch.from_numpy(y)


train_df = manifest_df[manifest_df['split'] == 'train'].reset_index(drop=True)
val_df = manifest_df[manifest_df['split'] == 'val'].reset_index(drop=True)
test_df = manifest_df[manifest_df['split'] == 'test'].reset_index(drop=True)

cache = VolumeCache(cache=True)
train_ds = RandomPatchDataset(train_df, cache, TRAIN_PATCHES_PER_EPOCH)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('train patches per epoch:', len(train_ds))


train/val/test: 99 10 11
train patches per epoch: 128


In [7]:
def build_mednext_model():
    model = create_mednext_v1(
        num_input_channels=1,
        num_classes=1,
        model_id=MEDNEXT_MODEL_ID,
        kernel_size=MEDNEXT_KERNEL_SIZE,
        deep_supervision=MEDNEXT_DEEP_SUPERVISION,
    )
    return model


model = build_mednext_model()
num_params = sum(p.numel() for p in model.parameters())
print(f'Official MedNeXt-{MEDNEXT_MODEL_ID} params:', f'{num_params/1e6:.2f}M')
print('kernel_size:', MEDNEXT_KERNEL_SIZE, 'deep_supervision:', MEDNEXT_DEEP_SUPERVISION)


Official MedNeXt-S params: 5.55M
kernel_size: 3 deep_supervision: False


In [8]:
def primary_logits(output):
    if isinstance(output, (list, tuple)):
        return output[0]
    return output


def dice_loss_with_logits(logits, targets, eps=1e-5):
    probs = torch.sigmoid(logits)
    dims = tuple(range(1, probs.ndim))
    inter = torch.sum(probs * targets, dim=dims)
    denom = torch.sum(probs, dim=dims) + torch.sum(targets, dim=dims)
    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def combined_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dl = dice_loss_with_logits(logits, targets)
    return 0.3 * bce + 0.7 * dl


def iter_starts(size, patch, overlap):
    if size <= patch:
        return [0]
    step = max(1, int(round(patch * (1.0 - overlap))))
    starts = list(range(0, size - patch + 1, step))
    last = size - patch
    if starts[-1] != last:
        starts.append(last)
    return starts


@torch.no_grad()
def sliding_window_predict_logits(model, image_zyx, device, patch_size=PATCH_SIZE, overlap=SLIDING_WINDOW_OVERLAP):
    model.eval()
    original_shape = image_zyx.shape
    image_pad, pad_after = pad_to_at_least(image_zyx, patch_size, pad_value=-1.0)
    zdim, ydim, xdim = image_pad.shape
    pz, py, px = patch_size
    z_starts = iter_starts(zdim, pz, overlap)
    y_starts = iter_starts(ydim, py, overlap)
    x_starts = iter_starts(xdim, px, overlap)

    logits_sum = np.zeros(image_pad.shape, dtype=np.float32)
    count = np.zeros(image_pad.shape, dtype=np.float32)

    for z in z_starts:
        for y in y_starts:
            for x in x_starts:
                patch = image_pad[z:z+pz, y:y+py, x:x+px]
                inp = torch.from_numpy(patch[None, None].astype(np.float32)).to(device)
                with autocast(enabled=(USE_AMP and device.type == 'cuda')):
                    logits = primary_logits(model(inp))
                logits_np = logits.detach().float().cpu().numpy()[0, 0]
                logits_sum[z:z+pz, y:y+py, x:x+px] += logits_np
                count[z:z+pz, y:y+py, x:x+px] += 1.0

    logits_avg = logits_sum / np.maximum(count, 1e-6)
    oz, oy, ox = original_shape
    return logits_avg[:oz, :oy, :ox]


@torch.no_grad()
def evaluate_rows(model, rows_df, device, max_cases=None):
    rows = rows_df.reset_index(drop=True)
    if max_cases is not None:
        rows = rows.iloc[:max_cases]
    dices = []
    for row in rows.to_dict('records'):
        item = cache.get(row)
        logits = sliding_window_predict_logits(model, item['image'], device)
        pred = (1.0 / (1.0 + np.exp(-logits)) >= THRESHOLD).astype(np.uint8)
        pred = remove_small_components(pred, POSTPROCESS_MIN_COMPONENT_SIZE)
        d = dice_binary(pred, item['label'])
        dices.append(d)
        print(f"val {row['case_id']}: Dice {d:.4f}")
    return float(np.mean(dices)) if dices else 0.0


In [9]:
RUN_TRAIN = True
RESUME_CHECKPOINT = None  # path neu muon resume

if RUN_TRAIN:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    scaler = GradScaler(enabled=(USE_AMP and device.type == 'cuda'))

    start_epoch = 1
    best_val_dice = -1.0
    history = []

    if RESUME_CHECKPOINT is not None:
        try:
            ckpt = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
        except TypeError:
            ckpt = torch.load(RESUME_CHECKPOINT, map_location=device)
        model.load_state_dict(ckpt['model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        scheduler.load_state_dict(ckpt['scheduler'])
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        best_val_dice = float(ckpt.get('best_val_dice', -1.0))
        print('Resumed from', RESUME_CHECKPOINT, 'start_epoch', start_epoch, 'best', best_val_dice)

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        t0 = time.time()
        model.train()
        losses = []
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(USE_AMP and device.type == 'cuda')):
                logits = primary_logits(model(images))
                loss = combined_loss(logits, labels)
            scaler.scale(loss).backward()
            if GRAD_CLIP_NORM is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        train_loss = float(np.mean(losses)) if losses else np.nan
        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'lr': optimizer.param_groups[0]['lr'],
            'val_dice': np.nan,
            'epoch_time_sec': time.time() - t0,
        }

        do_val = epoch == 1 or epoch == MAX_EPOCHS or (epoch % VALIDATE_EVERY == 0)
        if do_val:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            val_dice = evaluate_rows(model, val_df, device)
            row['val_dice'] = val_dice
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                best_path = CHECKPOINT_DIR / 'checkpoint_best.pth'
                torch.save({
                    'epoch': epoch,
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(),
                    'best_val_dice': best_val_dice,
                    'config': {
                        'MAX_EPOCHS': MAX_EPOCHS,
                        'PATCH_SIZE': PATCH_SIZE,
                        'MEDNEXT_REPO': str(MEDNEXT_REPO),
                        'MEDNEXT_MODEL_ID': MEDNEXT_MODEL_ID,
                        'MEDNEXT_KERNEL_SIZE': MEDNEXT_KERNEL_SIZE,
                        'CT_WINDOW': CT_WINDOW,
                    },
                }, best_path)
                print(f'New best checkpoint: {best_path} val_dice={best_val_dice:.4f}')

        latest_path = CHECKPOINT_DIR / 'checkpoint_latest.pth'
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_val_dice': best_val_dice,
        }, latest_path)

        history.append(row)
        pd.DataFrame(history).to_csv(OUTPUT_ROOT / '02_16_mednext_s_training_history.csv', index=False)
        print(f"Epoch {epoch:03d}/{MAX_EPOCHS} loss={train_loss:.4f} val={row['val_dice']} best={best_val_dice:.4f} time={row['epoch_time_sec']:.1f}s")

    print('Training done. Best val Dice:', best_val_dice)
else:
    print('Skipped training. Set RUN_TRAIN=True to train.')


/tmp/ipykernel_58/4250268075.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(USE_AMP and device.type == 'cuda'))
/tmp/ipykernel_58/4250268075.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):
/tmp/ipykernel_58/4125398824.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


val phe_0013: Dice 0.0000
val phe_0014: Dice 0.0000
val phe_0060: Dice 0.0000
val phe_0078: Dice 0.0023
val phe_0080: Dice 0.0000
val phe_0115: Dice 0.0000
val phe_0119: Dice 0.0000
val phe_0130: Dice 0.0000
val phe_0138: Dice 0.0000
val phe_0160: Dice 0.0054
New best checkpoint: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/checkpoints/checkpoint_best.pth val_dice=0.0008
Epoch 001/120 loss=0.7077 val=0.0007673398979110224 best=0.0008 time=69.9s


/tmp/ipykernel_58/4250268075.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


Epoch 002/120 loss=0.6828 val=nan best=0.0008 time=41.1s
Epoch 003/120 loss=0.6781 val=nan best=0.0008 time=30.8s
Epoch 004/120 loss=0.6793 val=nan best=0.0008 time=26.4s
Epoch 005/120 loss=0.6681 val=nan best=0.0008 time=27.0s
Epoch 006/120 loss=0.6670 val=nan best=0.0008 time=26.4s
Epoch 007/120 loss=0.6530 val=nan best=0.0008 time=26.1s
Epoch 008/120 loss=0.6453 val=nan best=0.0008 time=26.2s
Epoch 009/120 loss=0.6318 val=nan best=0.0008 time=26.3s


/tmp/ipykernel_58/4125398824.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):
/tmp/ipykernel_58/4125398824.py:72: RuntimeWarning: overflow encountered in exp
  pred = (1.0 / (1.0 + np.exp(-logits)) >= THRESHOLD).astype(np.uint8)


...

val phe_0013: Dice 0.3373
val phe_0014: Dice 0.3681
val phe_0060: Dice 0.4490
val phe_0078: Dice 0.4093
val phe_0080: Dice 0.4057
val phe_0115: Dice 0.2089
val phe_0119: Dice 0.0626
val phe_0130: Dice 0.2336
val phe_0138: Dice 0.2569
val phe_0160: Dice 0.1683
New best checkpoint: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/checkpoints/checkpoint_best.pth val_dice=0.2900
Epoch 110/120 loss=0.4609 val=0.28997202055813126 best=0.2900 time=26.1s


/tmp/ipykernel_58/4250268075.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):


Epoch 111/120 loss=0.4506 val=nan best=0.2900 time=26.2s
Epoch 112/120 loss=0.4371 val=nan best=0.2900 time=26.2s
Epoch 113/120 loss=0.4098 val=nan best=0.2900 time=26.2s
Epoch 114/120 loss=0.4330 val=nan best=0.2900 time=26.3s
Epoch 115/120 loss=0.4451 val=nan best=0.2900 time=26.2s
Epoch 116/120 loss=0.4402 val=nan best=0.2900 time=26.2s
Epoch 117/120 loss=0.4489 val=nan best=0.2900 time=26.2s
Epoch 118/120 loss=0.4538 val=nan best=0.2900 time=26.2s
Epoch 119/120 loss=0.4486 val=nan best=0.2900 time=26.2s


/tmp/ipykernel_58/4125398824.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):
/tmp/ipykernel_58/4125398824.py:72: RuntimeWarning: overflow encountered in exp
  pred = (1.0 / (1.0 + np.exp(-logits)) >= THRESHOLD).astype(np.uint8)


val phe_0013: Dice 0.3377
val phe_0014: Dice 0.3808
val phe_0060: Dice 0.4449
val phe_0078: Dice 0.4127
val phe_0080: Dice 0.4068
val phe_0115: Dice 0.2112
val phe_0119: Dice 0.0665
val phe_0130: Dice 0.2277
val phe_0138: Dice 0.2527
val phe_0160: Dice 0.1685
New best checkpoint: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/checkpoints/checkpoint_best.pth val_dice=0.2909
Epoch 120/120 loss=0.4394 val=0.2909346577113335 best=0.2909 time=26.2s
Training done. Best val Dice: 0.2909346577113335


In [10]:
RUN_TEST_INFERENCE = True
CHECKPOINT_FOR_TEST = CHECKPOINT_DIR / 'checkpoint_best.pth'

if RUN_TEST_INFERENCE:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = build_mednext_model().to(device)
    try:
        ckpt = torch.load(CHECKPOINT_FOR_TEST, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(CHECKPOINT_FOR_TEST, map_location=device)
    model.load_state_dict(ckpt['model'])
    print('Loaded checkpoint:', CHECKPOINT_FOR_TEST)
    print('Checkpoint epoch:', ckpt.get('epoch'), 'best_val_dice:', ckpt.get('best_val_dice'))

    metric_rows = []
    for row in test_df.to_dict('records'):
        print('Predicting', row['case_id'])
        item = cache.get(row)
        logits = sliding_window_predict_logits(model, item['image'], device)
        prob = 1.0 / (1.0 + np.exp(-logits))
        pred = (prob >= THRESHOLD).astype(np.uint8)
        pred = remove_small_components(pred, POSTPROCESS_MIN_COMPONENT_SIZE)

        out_path = PRED_DIR / f"{row['case_id']}.nii.gz"
        write_nifti_like(pred, item['image_obj'], out_path)
        metrics = metric_binary(pred, item['label'])
        metric_rows.append({
            'case_id': row['case_id'],
            'prediction_file': str(out_path),
            'reference_file': row['label_path'],
            **metrics,
        })
        print(row['case_id'], 'Dice', f"{metrics['Dice']:.4f}", 'IoU', f"{metrics['IoU']:.4f}")

    metrics_df = pd.DataFrame(metric_rows)
    per_case_csv = OUTPUT_ROOT / '02_16_mednext_s_phe_only_metrics_per_case.csv'
    metrics_df.to_csv(per_case_csv, index=False)

    summary_rows = []
    numeric_cols = [c for c in metrics_df.columns if c not in {'case_id', 'prediction_file', 'reference_file'}]
    for col in numeric_cols:
        vals = pd.to_numeric(metrics_df[col], errors='coerce').dropna()
        if len(vals) == 0:
            continue
        summary_rows.append({
            'label': 'PHE',
            'metric': col,
            'mean': float(vals.mean()),
            'median': float(vals.median()),
            'std': float(vals.std(ddof=0)),
            'n': int(len(vals)),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_csv = OUTPUT_ROOT / '02_16_mednext_s_phe_only_metrics_summary.csv'
    summary_df.to_csv(summary_csv, index=False)

    print('Wrote:', per_case_csv)
    print('Wrote:', summary_csv)
    display(summary_df)
else:
    print('Skipped test inference. Set RUN_TEST_INFERENCE=True after training.')


Loaded checkpoint: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/checkpoints/checkpoint_best.pth
Checkpoint epoch: 120 best_val_dice: 0.2909346577113335
Predicting phe_0002


/tmp/ipykernel_58/4125398824.py:52: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(USE_AMP and device.type == 'cuda')):
/tmp/ipykernel_58/4271867757.py:20: RuntimeWarning: overflow encountered in exp
  prob = 1.0 / (1.0 + np.exp(-logits))


phe_0002 Dice 0.4967 IoU 0.3304
Predicting phe_0029
phe_0029 Dice 0.3815 IoU 0.2357
Predicting phe_0033
phe_0033 Dice 0.4266 IoU 0.2711
Predicting phe_0051
phe_0051 Dice 0.2515 IoU 0.1439
Predicting phe_0061
phe_0061 Dice 0.1797 IoU 0.0987
Predicting phe_0084
phe_0084 Dice 0.5140 IoU 0.3459
Predicting phe_0095
phe_0095 Dice 0.3043 IoU 0.1794
Predicting phe_0096
phe_0096 Dice 0.5767 IoU 0.4052
Predicting phe_0113
phe_0113 Dice 0.1884 IoU 0.1040
Predicting phe_0116
phe_0116 Dice 0.2023 IoU 0.1125
Predicting phe_0167
phe_0167 Dice 0.0000 IoU 0.0000
Wrote: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/02_16_mednext_s_phe_only_metrics_per_case.csv
Wrote: /kaggle/working/outputs_02_16_kaggle_mednext_s_phe_only_baseline/02_16_mednext_s_phe_only_metrics_summary.csv


,label,metric,mean,median,std,n
0,PHE,Dice,3.201526e-01,3.042900e-01,0.167351,11
1,PHE,FN,1.395455e+03,1.185000e+03,1213.834747,11
2,PHE,FP,2.152818e+03,1.471000e+03,1679.737361,11
3,PHE,IoU,2.024406e-01,1.794470e-01,0.119602,11
4,PHE,TN,7.835884e+06,8.382508e+06,800296.619208,11
5,PHE,TP,1.056545e+03,7.790000e+02,1105.440049,11
6,PHE,n_pred,3.209364e+03,2.211000e+03,2254.657655,11
7,PHE,n_ref,2.452000e+03,1.635000e+03,2110.821512,11


## Compare note

So sanh truc tiep voi cac run cu tren cung test split 11 case:

- `02_15`: nnU-Net CT-only, 60 epoch.
- `02_12b`: nnU-Net CT-only, 120 epoch.
- `02_12`: nnU-Net CT-only, 250 epoch.
- `02_14`: nnU-Net CT + ICH teacher prior, 250 epoch.
- `02_16`: official MedNeXt-S CT-only, 120 epoch.

Neu `02_16` tot hon `02_12b`, kha nang backbone MedNeXt-S co loi. Neu khong, can uu tien multi-window preprocessing va post-processing truoc khi ket luan ve backbone.
